# Урок 9.4. Сравнение библиотек и выбор

**Интерактивная версия:** `lessons/lesson_9_4/web/index.html`

Полный бенчмарк (3 задачи × 3 библиотеки × 2 режима × 3 разбиения) выполняет скрипт
`python lessons/lesson_9_4/examples/benchmark.py` — около 5 минут, в основном из-за CatBoost на категориях.
Скрипт сохраняет результаты в `web/results.js`; здесь мы их загружаем и разбираем, а затем
повторяем облегчённую версию протокола и проверяем странное время CatBoost.

In [1]:
import json
import sys
import time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "python" / "gbcourse").is_dir())
sys.path.insert(0, str(ROOT / "shared" / "python"))

import catboost as cb
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from gbcourse import datasets

## 1. Результаты полного бенчмарка

In [2]:
text = (ROOT / "lessons" / "lesson_9_4" / "web" / "results.js").read_text(encoding="utf-8")
start = text.index(".benchmark = ") + len(".benchmark = ")
bench = json.loads(text[start: text.rindex(";\n})")])
print("версии:", bench["versions"])
rows = [{"задача": t["name"].split(",")[0], "метрика": t["metric"], **r} for t in bench["tasks"] for r in t["rows"]]
table = pd.DataFrame(rows).pivot_table(index=["задача", "lib"], columns="mode", values=["score", "time"], sort=False)
table.round(4)

версии: {'XGBoost': '3.4.1', 'LightGBM': '4.7.0', 'CatBoost': '1.2.10'}


score            time        
mode                                  default   early default   early
задача                       lib                                     
Фридман                      XGBoost   1.4163  1.2765   0.089   0.309
                             LightGBM  1.2708  1.2479   0.073   0.222
                             CatBoost  1.0840  1.0734   0.984   0.503
«Луны» + 8 шумовых признаков XGBoost   0.2411  0.2106   0.062   0.151
                             LightGBM  0.2164  0.2125   0.074   0.143
                             CatBoost  0.2124  0.2089   3.816   1.239
Категория из 100 значений    XGBoost   1.1972  1.0718   0.049   0.085
                             LightGBM  1.0818  1.0513   0.054   0.098
                             CatBoost  1.1011  1.0859  68.696  30.144

## 2. Облегчённый протокол: XGBoost и LightGBM на задаче Фридмана

Разбиения те же, что в бенчмарке (перестановка с тем же зерном, 3000/1000/1000), поэтому числа должны совпасть.

In [3]:
X, y = datasets.friedman1(n=5000, noise=1.0, seed=120)
res = []
for seed in range(3):
    perm = np.random.default_rng(seed).permutation(len(y))
    tr, va, te = perm[:3000], perm[3000:4000], perm[4000:]
    rmse = lambda p, te=te: float(np.sqrt(np.mean((y[te] - p) ** 2)))
    m = xgb.XGBRegressor(n_estimators=5000, learning_rate=0.05, early_stopping_rounds=100, random_state=seed)
    m.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=False)
    res.append({"разбиение": seed, "библиотека": "XGBoost", "RMSE": rmse(m.predict(X[te])), "деревьев": m.best_iteration + 1})
    m = lgb.LGBMRegressor(n_estimators=5000, learning_rate=0.05, verbose=-1, random_state=seed)
    m.fit(X[tr], y[tr], eval_X=(X[va],), eval_y=(y[va],), callbacks=[lgb.early_stopping(100, verbose=False)])
    res.append({"разбиение": seed, "библиотека": "LightGBM", "RMSE": rmse(m.predict(X[te])), "деревьев": m.best_iteration_})
res = pd.DataFrame(res)
res.groupby("библиотека")[["RMSE", "деревьев"]].agg(["mean", "std"]).round(3)

RMSE        деревьев         
             mean    std     mean      std
библиотека                                
LightGBM    1.248  0.046    277.0   73.546
XGBoost     1.276  0.044    422.0  120.079

Сравните средние с разбросом между разбиениями, прежде чем делать вывод.

## 3. Время CatBoost с категориальным признаком

В бенчмарке CatBoost с категорией обучался в сотни раз дольше. Проверим, от чего зависит время:
от числа строк или от числа деревьев.

In [4]:
x, cat, yc = datasets.categorical_regression(n=5000, n_categories=100, noise=1.0, seed=122)
rows = []
for n in (1000, 3000):
    for it in (20, 200):
        for with_cat in (True, False):
            df = pd.DataFrame({"x": x, "c": cat.astype(str) if with_cat else cat})[:n]
            t0 = time.perf_counter()
            cb.CatBoostRegressor(iterations=it, verbose=0, cat_features=["c"] if with_cat else None).fit(df, yc[:n])
            rows.append({"строк": n, "деревьев": it, "категория как": "категория" if with_cat else "число",
                         "время, с": time.perf_counter() - t0})
pd.DataFrame(rows).pivot_table(index=["строк", "деревьев"], columns="категория как", values="время, с").round(2)

категория как   категория  число
строк деревьев                  
1000  20             1.37   0.06
      200           14.65   0.17
3000  20             1.48   0.07
      200           13.48   0.19

Время почти не зависит от числа строк и растёт с числом деревьев: в нашем окружении (CatBoost 1.2.10)
на каждое дерево уходит порядка десятков миллисекунд накладных расходов при наличии категориального признака.
На другой машине и версии цифры могут отличаться — поэтому время всегда измеряют на своих данных.

## Упражнения

1. Добавьте в облегчённый протокол CatBoost (с ранней остановкой) и сравните с результатами бенчмарка.
2. Повторите раздел 2 с `learning_rate=0.1`. Изменился ли вывод о лидере?

Решения: `python lessons/lesson_9_4/exercises/solutions.py`.